# Part 1: Gnostic Metrics Step by Step

In this notebook, we compare classical metrics with Machine Gnostics metrics on all 4 Anscombe datasets.

Learning goals:
- compute mean and median (NumPy vs Machine Gnostics)
- compare correlation, $R^2$, and RMSE
- understand where values differ and why

In [1]:
import numpy as np
import pandas as pd
from scipy import stats

from machinegnostics.data import make_anscombe_check_data
from machinegnostics.metrics import mean as mg_mean, median as mg_median, correlation as mg_correlation
from machinegnostics.metrics import robr2, root_mean_squared_error
from machinegnostics.models import LinearRegressor

from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error

## Step 1: Load Data

In [2]:
datasets = {}
for ds_id in [1, 2, 3, 4]:
    x, y = make_anscombe_check_data(ds_id)
    datasets[ds_id] = {"x": np.asarray(x, dtype=float), "y": np.asarray(y, dtype=float)}

print({k: len(v['x']) for k, v in datasets.items()})

{1: 11, 2: 11, 3: 11, 4: 11}


## Step 2: Define Metric Helpers

In [3]:
def compute_classical_metrics(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    x_2d = x.reshape(-1, 1)

    model = LinearRegression().fit(x_2d, y)
    y_pred = model.predict(x_2d)

    return {
        "mean_x": float(np.mean(x)),
        "mean_y": float(np.mean(y)),
        "median_x": float(np.median(x)),
        "median_y": float(np.median(y)),
        "corr": float(np.corrcoef(x, y)[0, 1]),
        "r2": float(r2_score(y, y_pred)),
        "rmse": float(np.sqrt(mean_squared_error(y, y_pred))),
        "y_pred": y_pred,
    }


def compute_mg_metrics(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    model = LinearRegressor(max_iter=300, early_stopping=True, tolerance=1e-6, mg_loss="hi", history=True, verbose=False)
    model.fit(x, y)
    y_pred = np.asarray(model.predict(x), dtype=float)

    return {
        "mean_x": float(mg_mean(x)),
        "mean_y": float(mg_mean(y)),
        "median_x": float(mg_median(x)),
        "median_y": float(mg_median(y)),
        "corr": float(mg_correlation(x, y)),
        "r2": float(r2_score(y, y_pred)),
        "robr2": float(robr2(y, y_pred, w=getattr(model, "weights", None))),
        "rmse": float(root_mean_squared_error(y, y_pred)),
        "y_pred": y_pred,
        "model": model,
    }

## Step 3: Compute Metrics for All Datasets

In [4]:
metrics_all = {}
for ds_id in [1, 2, 3, 4]:
    x = datasets[ds_id]["x"]
    y = datasets[ds_id]["y"]
    metrics_all[ds_id] = {
        "x": x,
        "y": y,
        "classical": compute_classical_metrics(x, y),
        "mg": compute_mg_metrics(x, y),
    }

print("Computed classical and machine gnostics metrics for datasets 1-4.")

Computed classical and machine gnostics metrics for datasets 1-4.


## Step 4: Build a Comparison Table

In [5]:
rows = []
for ds_id in [1, 2, 3, 4]:
    cls = metrics_all[ds_id]["classical"]
    mg = metrics_all[ds_id]["mg"]

    rows.append({
        "Dataset": ds_id,
        "Mean(np)": cls["mean_y"],
        "Mean(mg)": mg["mean_y"],
        "Median(np)": cls["median_y"],
        "Median(mg)": mg["median_y"],
        "Corr(np)": cls["corr"],
        "Corr(mg)": mg["corr"],
        "R2(np)": cls["r2"],
        "R2(mg)": mg["r2"],
        "RobR2(mg)": mg["robr2"],
        "RMSE(np)": cls["rmse"],
        "RMSE(mg)": mg["rmse"],
    })

comparison_df = pd.DataFrame(rows)
comparison_df.round(4)

,Dataset,Mean(np),Mean(mg),Median(np),Median(mg),Corr(np),Corr(mg),R2(np),R2(mg),RobR2(mg),RMSE(np),RMSE(mg)
0,1,7.5009,7.7474,7.58,7.5895,0.8164,0.9875,0.6665,0.6652,0.9967,1.1185,0.4670
1,2,7.5009,8.6440,8.14,8.0773,0.8162,0.9627,0.6662,0.6425,0.8960,1.1191,0.8933
2,3,7.5000,6.9335,7.11,7.1687,0.8163,0.9886,0.6663,0.5631,1.0000,1.1183,0.0005
3,4,7.5009,7.2075,7.04,7.2220,0.8165,0.5582,0.6667,0.6651,0.9978,1.1177,0.7184


## Step 5: Single-Dataset Walkthrough (Dataset 1)
This is useful for beginners who want to inspect one dataset deeply before moving on.

In [6]:
ds_id = 1
cls = metrics_all[ds_id]["classical"]
mg = metrics_all[ds_id]["mg"]

print(f"Dataset {ds_id} walkthrough")
print("-" * 70)
print(f"Mean y        -> numpy: {cls['mean_y']:.6f}, machinegnostics: {mg['mean_y']:.6f}")
print(f"Median y      -> numpy: {cls['median_y']:.6f}, machinegnostics: {mg['median_y']:.6f}")
print(f"Correlation   -> numpy: {cls['corr']:.6f}, machinegnostics: {mg['corr']:.6f}")
print(f"R2            -> numpy/sklearn: {cls['r2']:.6f}, machinegnostics model fit: {mg['r2']:.6f}")
print(f"RMSE          -> numpy/sklearn: {cls['rmse']:.6f}, machinegnostics: {mg['rmse']:.6f}")
print(f"RobR2 (mg)    -> {mg['robr2']:.6f}")

Dataset 1 walkthrough
----------------------------------------------------------------------
Mean y        -> numpy: 7.500909, machinegnostics: 7.747400
Median y      -> numpy: 7.580000, machinegnostics: 7.589480
Correlation   -> numpy: 0.816421, machinegnostics: 0.987515
R2            -> numpy/sklearn: 0.666542, machinegnostics model fit: 0.665166
RMSE          -> numpy/sklearn: 1.118550, machinegnostics: 0.466973
RobR2 (mg)    -> 0.996684


### Key takeaway
Anscombe datasets look similar in classical summary values, but later notebooks will show why their distribution shape and regression behavior are very different.